# Install Library

In [20]:
!pip install kafka-python

# Import Library yang dibutuhkan

In [21]:
import psycopg2
from kafka import KafkaProducer
import json
import time

# Buat Koneksi ke Database

In [22]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="streaming_db",
    user="airflow",
    password="airflow"
)
print("Connected to PostgreSQL")

Connected to PostgreSQL


# Tes Kafka

In [23]:
producer = KafkaProducer(
    bootstrap_servers="kafka:29092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

producer.send("crypto_stream", {"coin": "BTC", "price": 30000})
producer.flush()
print("✅ berhasil kirim")

✅ berhasil kirim


# Membaca data PostgreSQL dan kirim ke Kafka secara berulang

In [24]:
producer = KafkaProducer(
    bootstrap_servers="kafka:29092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

cursor = conn.cursor()
last_id = 0

while True:
    cursor.execute(
        "SELECT id, coin, price, time FROM crypto_price WHERE id > %s ORDER BY id ASC",
        (last_id,)
    )
    rows = cursor.fetchall()

    for row in rows:
        last_id = row[0]
        producer.send("crypto_stream", {
            "coin": row[1],
            "price": float(row[2]),
            "time": str(row[3])
        })

    producer.flush()
    time.sleep(3)

KeyboardInterrupt: 

Cell while True berjalan terus-menerus hingga dihentikan secara manual.

# Pemeriksaan akhir Notebook